# 🌌 Live coding: MLflow y la Biblioteca del Templo Jedi

**Duración aproximada:** 10–15 minutos  
**Objetivo:** registrar una ejecución real de Machine Learning y comprobar qué guarda MLflow.

En esta demostración utilizaremos:

- el conjunto de datos **Digits**, con imágenes de números escritos a mano;
- un modelo **Random Forest**;
- **MLflow Tracking** para registrar parámetros, métricas, un artefacto y el modelo;
- una segunda ejecución opcional para comparar resultados.

> **Idea que debe quedar clara:** Scikit-learn entrena el modelo. MLflow conserva la historia de cómo se entrenó y qué resultados obtuvo.

---

**Ruta de la píldora:** demo guiada → reto `03_Orden_66_RETO.ipynb` → corrección con `02_Archivos_Jedi_SOLUCION.ipynb`.
**Entorno validado:** Python 3.12, MLflow 3.14.0, scikit-learn 1.9.0, pandas 2.3.3 y matplotlib 3.11.0.

> Esta versión conserva los huecos exigidos por la guía de entrega. Para exponer, usa `Live_Coding_DEMO_SEGURA.ipynb`.

> Esta versión conserva los huecos exigidos por la guía de entrega. Para exponer, usa `Live_Coding_DEMO_SEGURA.ipynb`.


> **Modo alumnado:** las celdas de instalación e importaciones están preparadas. En los bloques de trabajo hay aproximadamente un 40% de pistas y un 60% de huecos. Completa los bloques TODO/COMPLETAR antes de ejecutar todo el notebook.


## 🗺️ Mapa de la demostración

Durante el live coding seguiremos este recorrido:

1. Instalamos las librerías.
2. Importamos las herramientas.
3. Cargamos y dividimos los datos.
4. Creamos el experimento: nuestra **campaña Jedi**.
5. Abrimos una ejecución: nuestra **misión Jedi**.
6. Entrenamos el modelo y registramos la información.
7. Consultamos el expediente que ha guardado MLflow.
8. Creamos una segunda misión y comparamos ambas.

### Frase para introducir el código

> “Ahora vamos a comprobar que la analogía de la Biblioteca Jedi no es solamente una historia. Vamos a registrar una misión real y veremos qué conserva MLflow de ella.”

# 1. Preparar el entorno

Google Colab no incluye MLflow de forma predeterminada, por lo que primero instalamos la librería.

### Qué explico mientras se ejecuta

> “Esta celda no forma parte del experimento. Solo prepara el entorno para poder utilizar MLflow. La opción `-q` hace que la instalación muestre menos mensajes en pantalla.”

In [ ]:
# Entorno validado el 15-07-2026. Ejecuta esta celda una sola vez en Colab.
%pip install -q "mlflow==3.14.0" "scikit-learn==1.9.0" "pandas==2.3.3" "matplotlib==3.11.0"

# 2. Importar las herramientas

Importamos MLflow, las funciones de Scikit-learn y las utilidades necesarias para evaluar el modelo.

### Qué explico

> “Aquí todavía no estamos entrenando ni registrando nada. Solo estamos preparando las herramientas que necesitaremos. Importamos `mlflow` y también `mlflow.sklearn`, porque nuestro modelo pertenece a Scikit-learn.”

In [ ]:
# Librerías generales
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# MLflow y su integración con Scikit-learn
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# Dataset, modelo y métricas
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

print("MLflow:", mlflow.__version__)

# 3. Preparar los datos y decidir la estrategia

Utilizaremos **Digits** porque es un conjunto visual, cercano y rápido de entrenar. Contiene imágenes de números escritos a mano del 0 al 9. Cada imagen mide 8 × 8 píxeles y se transforma en 64 características.

En la analogía:

- los **datos** son la información de la misión;
- `n_estimators`, `max_depth` y `random_state` son parte de la **estrategia**;
- esa estrategia se decide antes de entrenar, por lo que se registrará como **parámetros**.

### Qué explico

> “Nuestro modelo recibirá imágenes de números escritos a mano y tendrá que reconocer qué dígito aparece. Primero preparamos los datos y decidimos cómo será el modelo. Todavía no sabemos si funcionará bien, por eso estos valores son decisiones previas y no métricas.”

In [ ]:
# Cargamos Digits como DataFrame.
digits = load_digits(as_frame=True)
# TODO: define X y y a partir de digits.

# Divide los datos en train/test con test_size=0.20, random_state=42 y stratify=y.
X_train, X_test, y_train, y_test = train_test_split(
    # COMPLETAR
)

# Estrategia de Coruscant: 100 árboles, profundidad 10 y random_state=42.
parametros = {
    # COMPLETAR
}


# 4. Crear la Biblioteca y la campaña

MLflow necesita saber dónde guardará la información.

- La **Tracking URI** es la dirección de la Biblioteca.
- El **Experiment** agrupa las ejecuciones relacionadas con un mismo objetivo.
- Cada entrenamiento posterior será una **Run** dentro de ese experimento.

En este Colab utilizamos una base SQLite local para los metadatos.

### Qué explico

> “Con `set_tracking_uri` indicamos dónde estará nuestra Biblioteca. Después, con `set_experiment`, abrimos una campaña llamada `Biblioteca_Templo_Jedi_Digits`. Esta línea no entrena nada; solamente organiza dónde quedarán registradas las misiones.”

In [ ]:
# Configura la Biblioteca Jedi con SQLite portátil.
TRACKING_DB = Path("mlflow_jedi_digits.db").resolve()
# TODO: construye TRACKING_URI con formato sqlite:///...
TRACKING_URI = ""
NOMBRE_EXPERIMENTO = "Biblioteca_Templo_Jedi_Digits_Live"

# Selecciona el tracking y crea/recupera el Experiment.
# COMPLETAR set_tracking_uri y set_experiment


# 5. Abrir una misión, entrenar y registrar

Esta es la celda principal del live coding.

Dentro de `mlflow.start_run()`:

1. creamos el modelo;
2. lo entrenamos;
3. calculamos las métricas;
4. registramos parámetros y métricas;
5. guardamos una matriz de confusión como artefacto;
6. guardamos el modelo entrenado.

### Diferencias que conviene repetir

- **Parameter:** decisión conocida antes del entrenamiento.
- **Metric:** resultado numérico calculado después.
- **Artifact:** archivo generado por la ejecución.
- **Model:** objeto entrenado que podremos reutilizar.

### Frase antes de ejecutar

> “Con `start_run` abrimos el expediente de una misión. Todo lo que registremos dentro de este bloque quedará asociado a la misma ejecución.”

In [ ]:
# Abre una Run llamada Mision_Coruscant.
with mlflow.start_run(run_name="Mision_Coruscant") as run:
    # Crea y entrena el Random Forest.
    modelo = None
    # COMPLETAR fit(X_train, y_train)

    # Predice X_test y calcula cuatro métricas macro.
    predicciones = None
    metricas = {}
    # COMPLETAR

    # Registra parámetros, métricas y etiquetas.
    # COMPLETAR log_params, log_metrics y set_tags

    # Genera y guarda matriz_confusion_coruscant.png.
    ruta_matriz = Path("matriz_confusion_coruscant.png")
    # COMPLETAR gráfico, savefig y log_artifact

    # Firma y modelo MLflow.
    firma = infer_signature(X_train, modelo.predict(X_train))
    # COMPLETAR log_model con artifact_path=modelo_jedi
    run_id = run.info.run_id


# 6. Abrir el expediente que ha creado MLflow

No necesitamos la interfaz gráfica para comprobar el registro. Podemos recuperar la ejecución desde Python.

### Qué explico

> “El entrenamiento ha terminado, pero la información no ha desaparecido. MLflow conserva el nombre de la misión, su estado, los parámetros, las métricas, las etiquetas y los archivos que se han generado.”

In [ ]:
# Recupera la Run mediante run_id.
expediente = mlflow.get_run(run_id)
print("Nombre de la misión:", expediente.data.tags.get("mlflow.runName"))
print("Estado:", expediente.info.status)

# Recorre params y metrics para mostrar el expediente.
# COMPLETAR

# Lista los artefactos con MlflowClient.
cliente = MlflowClient()
# COMPLETAR list_artifacts(run_id)


# 7. Segunda misión opcional

Esta parte puede estar escrita de antemano. Solo hay que ejecutarla si queda tiempo.

Cambiaremos la estrategia para demostrar que un **Experiment** puede contener varias **Runs**. En esta segunda misión utilizamos menos árboles y una profundidad algo menor.

### Qué explico

> “No estamos sustituyendo la primera misión. Estamos creando otra ejecución dentro de la misma campaña. MLflow conservará ambas para que podamos compararlas.”

In [ ]:
# Segunda misión: Tatooine, más ligera.
parametros_tatooine = {
    "n_estimators": 30,
    # COMPLETAR max_depth=2 y random_state=42
}

with mlflow.start_run(run_name="Mision_Tatooine") as segunda_run:
    # Repite el patrón: crear, entrenar, predecir, medir y registrar.
    # COMPLETAR
    metricas_tatooine = {}
    # COMPLETAR


# 8. Comparar las misiones

`mlflow.search_runs()` recupera las ejecuciones de un experimento y devuelve una tabla.

### Qué explico

> “Aquí aparece el verdadero valor de MLflow. Ya no tenemos dos modelos sueltos ni dos resultados apuntados en lugares distintos. Tenemos dos expedientes comparables, con sus decisiones y sus resultados en la misma tabla.”

In [ ]:
# Compara las dos Runs del mismo Experiment.
comparacion = mlflow.search_runs(
    experiment_names=[NOMBRE_EXPERIMENTO],
    # COMPLETAR order_by para ordenar por accuracy
)

# Selecciona nombre, parámetros, métricas y estado.
columnas = [
    "tags.mlflow.runName",
    # COMPLETAR columnas restantes
]

# Renombra las columnas para presentar el informe.
# COMPLETAR


# 9. Cierre del live coding

## Lo que acabamos de hacer

| Biblioteca Jedi | MLflow | En el código |
|---|---|---|
| Biblioteca | Tracking URI | `mlflow.set_tracking_uri()` |
| Campaña | Experiment | `mlflow.set_experiment()` |
| Misión | Run | `mlflow.start_run()` |
| Estrategia | Parameters | `mlflow.log_params()` |
| Informe | Metrics | `mlflow.log_metrics()` |
| Holocrón | Artifact | `mlflow.log_artifact()` |
| Caballero entrenado | Model | `mlflow.sklearn.log_model()` |

### Cierre oral

> “Scikit-learn ha entrenado el modelo. MLflow no ha mejorado su accuracy ni ha decidido los parámetros. Lo que ha hecho es conservar la historia completa: qué probamos, qué resultado obtuvimos, qué archivo generamos y qué modelo quedó entrenado. Esa es la función de la Biblioteca del Templo Jedi: que ninguna misión y ningún aprendizaje se pierdan.”

---

## 🛟 Plan B para la exposición

- Si la instalación tarda, deja ejecutada la primera celda antes de empezar.
- Si aparece un aviso al guardar el modelo, explica que es una advertencia y comprueba que la celda termina con el mensaje de misión registrada.
- Si no da tiempo a crear la segunda misión, muestra solamente la primera y continúa con el Colab del ejercicio.
- Si reinicias el entorno de Colab, tendrás que volver a ejecutar las celdas desde el principio.
- Los valores exactos de las métricas pueden variar si se modifican los datos, la división o los parámetros.